<a href="https://colab.research.google.com/github/sanodyaJinadasa/DepreScope/blob/main/Copy_of_Explainable_Depression_Detection_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explainable Depression Detection from Social Media Text

**Research title:** Explainable Depression Detection from Social Media Text Using Fine-Tuned Transformer Models

This notebook implements a memory-conscious research pipeline:

1. Load `train.csv`, `val.csv`, and `test.csv`
2. Validate and clean the text/labels
3. Build a **TF-IDF + LinearSVC** classical baseline on a manageable stratified subset
4. Evaluate the baseline with Accuracy, Precision, Recall, F1 and ROC-AUC where available
5. Fine-tune a pretrained Transformer
6. Evaluate the Transformer
7. Generate SHAP explanations for selected test examples

> **Important:** This is a research text-classification system, not a clinical diagnostic tool. The labels and dataset construction determine what the model learns.


# Part 0 — Data Acquisition, Cleaning, and Train/Val/Test Split

This section downloads the raw Kaggle dataset, cleans it, and produces
`train.csv`, `val.csv`, and `test.csv` — the files the rest of this notebook expects.

Run this section **once per fresh Colab runtime**. If `train.csv`/`val.csv`/`test.csv`
already exist in your Colab session, you can skip straight to Part A below.

## 0.1 Install packages and set up Kaggle credentials

Upload your `kaggle.json` API key when prompted.

In [ ]:
# Install required packages (safe to re-run)
!pip -q install kaggle transformers datasets accelerate evaluate shap scikit-learn pandas numpy matplotlib seaborn

import os
import re
import gc
import pandas as pd
from google.colab import files

print("Please upload your kaggle.json API key file when prompted below.")
files.upload()

# Set up Kaggle credentials directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle credentials configured.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
Please upload your kaggle.json API key file when prompted below.


Saving kaggle.json to kaggle.json
Kaggle credentials configured.


## 0.2 Download and load the raw dataset

In [ ]:
# Download and unzip the dataset directly into Colab
!kaggle datasets download -d rishabhkausish/reddit-depression-dataset
!unzip -o -q reddit-depression-dataset.zip

# Confirm the actual extracted filename before loading
!ls -la *.csv

RAW_CSV_PATH = "reddit_depression_dataset.csv"  # <-- update if the ls output above shows a different name

df = pd.read_csv(RAW_CSV_PATH)
print("\nRaw shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Dataset URL: https://www.kaggle.com/datasets/rishabhkausish/reddit-depression-dataset
License(s): CC0-1.0
100% 431M/431M [00:10<00:00, 44.1MB/s]

-rw-r--r-- 1 root root 1165277530 Sep 10  2024 reddit_depression_dataset.csv


/tmp/ipykernel_1872/2729885661.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_CSV_PATH)



Raw shape: (2470778, 8)
Columns: ['Unnamed: 0', 'subreddit', 'title', 'body', 'upvotes', 'created_utc', 'num_comments', 'label']


,Unnamed: 0,subreddit,title,body,upvotes,created_utc,num_comments,label
0,47951,DeepThoughts,Deep thoughts underdog,"Only when we start considering ourselves, the ...",4.0,1.405309e+09,NaN,0.0
1,47952,DeepThoughts,"I like this sub, there's only two posts yet I ...",Anyway: Human Morality is a joke so long as th...,4.0,1.410568e+09,1.0,0.0
2,47957,DeepThoughts,Rebirth!,Hello. \nI am the new guy in charge here (Besi...,6.0,1.416458e+09,1.0,0.0
3,47959,DeepThoughts,"""I want to be like water. I want to slip throu...",NaN,25.0,1.416512e+09,2.0,0.0
4,47960,DeepThoughts,Who am I?,You could take any one cell in my body and kil...,5.0,1.416516e+09,4.0,0.0


## 0.3 Handle missing values

Missing `label` rows are dropped outright (no usable ground truth). Rows missing both `title` and `body` are dropped. Remaining missing text fields are filled with empty strings so they can still be combined safely.

In [ ]:
print("Missing values before handling:")
print(df.isnull().sum())

# 1. Drop rows with missing label — can't train/evaluate without ground truth
before = len(df)
df = df.dropna(subset=['label']).reset_index(drop=True)
print(f"\nDropped {before - len(df)} rows with missing label")

# 2. Drop rows missing BOTH title and body (no usable text at all)
before = len(df)
df = df.dropna(subset=['title', 'body'], how='all').reset_index(drop=True)
print(f"Dropped {before - len(df)} rows missing both title and body")

# 3. Fill remaining missing text fields
df['title'] = df['title'].fillna("")
df['body'] = df['body'].fillna("")

# 4. Fill/handle missing metadata (not used by the model, kept for EDA completeness)
for col in ['upvotes', 'num_comments']:
    if col in df.columns and df[col].isnull().sum() > 0:
        print(f"Filling {df[col].isnull().sum()} missing values in {col} with 0")
        df[col] = df[col].fillna(0)

if 'created_utc' in df.columns and df['created_utc'].isnull().sum() > 0:
    before = len(df)
    df = df.dropna(subset=['created_utc']).reset_index(drop=True)
    print(f"Dropped {before - len(df)} rows with missing created_utc")

# 5. Ensure label is a clean integer type
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print("\nMissing values after handling:")
print(df[['label', 'title', 'body']].isnull().sum())
print("\nShape after missing-value handling:", df.shape)


Missing values before handling:
Unnamed: 0           3
subreddit           20
title               23
body            461051
upvotes             64
created_utc        106
num_comments    113977
label              106
dtype: int64

Dropped 106 rows with missing label
Dropped 1 rows missing both title and body
Filling 113870 missing values in num_comments with 0

Missing values after handling:
label    0
title    0
body     0
dtype: int64

Shape after missing-value handling: (2470671, 8)


## 0.4 Combine text and inspect class balance / subreddit distribution

In [ ]:
df['text'] = (df['title'] + " " + df['body']).str.strip()
df['text_length'] = df['text'].str.len()

print("Label distribution:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True))

if 'subreddit' in df.columns:
    print("\nSubreddit distribution:")
    print(df['subreddit'].value_counts())

print("\nText length stats:")
print(df['text_length'].describe())


Label distribution:
label
0    1990260
1     480411
Name: count, dtype: int64
label
0    0.805554
1    0.194446
Name: proportion, dtype: float64

Subreddit distribution:
subreddit
teenagers       1956488
depression       290049
SuicideWatch     190362
happy             24609
DeepThoughts       9163
Name: count, dtype: int64

Text length stats:
count    2.470671e+06
mean     4.252188e+02
std      9.235704e+02
min      1.000000e+00
25%      7.500000e+01
50%      1.550000e+02
75%      4.190000e+02
max      6.485900e+04
Name: text_length, dtype: float64


## 0.5 Clean text, remove short/duplicate posts

This produces the `text_clean` column that the rest of the notebook (Part A onward) expects.

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)          # remove URLs
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text) # remove markdown links, keep link text
    text = re.sub(r'[*_~`>#]', ' ', text)                    # remove markdown symbols
    text = re.sub(r'&amp;|&gt;|&lt;', ' ', text)             # remove HTML entities
    text = re.sub(r'\s+', ' ', text).strip()                # collapse whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)

# Drop empty/very short posts after cleaning (need at least a few real words)
before = len(df)
df = df[df['text_clean'].str.len() >= 10].reset_index(drop=True)
print(f"Remaining rows after length filter: {len(df)} (dropped {before - len(df)})")

# Drop exact duplicate posts
before = len(df)
df = df.drop_duplicates(subset='text_clean').reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate posts")

print("\nFinal label distribution:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True))


Remaining rows after length filter: 2465488 (dropped 5183)
Dropped 861 duplicate posts

Final label distribution:
label
0    1984473
1     480154
Name: count, dtype: int64
label
0    0.805182
1    0.194818
Name: proportion, dtype: float64


## 0.6 Stratified train / validation / test split

70% train, 15% validation, 15% test, preserving class balance in each split.

In [ ]:
from sklearn.model_selection import train_test_split

train_df_full, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=42
)
val_df_full, test_df_full = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42
)

print(f"Train: {len(train_df_full)}  Val: {len(val_df_full)}  Test: {len(test_df_full)}")

train_df_full.to_csv("train.csv", index=False)
val_df_full.to_csv("val.csv", index=False)
test_df_full.to_csv("test.csv", index=False)
print("\nSaved train.csv, val.csv, test.csv")

# Free memory before Part A/B begin
del df, train_df_full, val_df_full, test_df_full, temp_df
gc.collect()
print("Raw dataframes released from memory.")


Train: 1725238  Val: 369694  Test: 369695

Saved train.csv, val.csv, test.csv
Raw dataframes released from memory.


---

# Part A onward — Baseline, Transformer, and Explainability

The cells below are unchanged from your reviewed notebook and now have the `train.csv` / `val.csv` / `test.csv` files they expect.

In [ ]:
# Optional: run this cell once in a fresh Colab runtime
# !pip -q install -U transformers datasets accelerate evaluate shap scikit-learn pandas numpy matplotlib seaborn

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cpu


## 1. Configuration

The key memory-control setting is `BASELINE_TRAIN_SIZE`.

Do **not** transform millions of Reddit posts into a TF-IDF matrix just because the original dataset is large. The baseline experiment can use a stratified subset while validation/test remain separate.


In [ ]:
# File paths
TRAIN_PATH = "train.csv"
VAL_PATH = "val.csv"
TEST_PATH = "test.csv"

TEXT_COLUMN = "text_clean"
LABEL_COLUMN = "label"

# Memory-safe baseline
BASELINE_TRAIN_SIZE = 200_000

# TF-IDF settings
TFIDF_MAX_FEATURES = 20_000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 5
TFIDF_MAX_DF = 0.95

# Transformer settings
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
TRANSFORMER_TRAIN_SIZE = 100_000
TRANSFORMER_VAL_SIZE = 20_000
TRANSFORMER_TEST_SIZE = 20_000

print("Configuration loaded.")


Configuration loaded.


## 2. Load only the required CSV columns

`usecols` reduces memory compared with loading every column. The text is explicitly converted to string and missing values are removed safely.


In [ ]:
required_columns = [TEXT_COLUMN, LABEL_COLUMN]

train_df = pd.read_csv(TRAIN_PATH, usecols=required_columns)
val_df = pd.read_csv(VAL_PATH, usecols=required_columns)
test_df = pd.read_csv(TEST_PATH, usecols=required_columns)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())
print("\nTrain labels:")
print(train_df[LABEL_COLUMN].value_counts(dropna=False))


Train: (1725238, 2)
Validation: (369694, 2)
Test: (369695, 2)

Columns: ['label', 'text_clean']

Train labels:
label
0    1389130
1     336108
Name: count, dtype: int64


In [ ]:
def prepare_dataframe(df, text_col=TEXT_COLUMN, label_col=LABEL_COLUMN):
    df = df[[text_col, label_col]].copy()

    # Remove missing values
    df = df.dropna(subset=[text_col, label_col])

    # Convert text safely
    df[text_col] = df[text_col].astype(str).str.strip()

    # Remove empty text
    df = df[df[text_col].str.len() > 0]

    # Normalize labels to integer if possible
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df.dropna(subset=[label_col])
    df[label_col] = df[label_col].astype(int)

    return df.reset_index(drop=True)

train_df = prepare_dataframe(train_df)
val_df = prepare_dataframe(val_df)
test_df = prepare_dataframe(test_df)

print("After cleaning:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


After cleaning:
Train: (1725238, 2)
Validation: (369694, 2)
Test: (369695, 2)


## 3. Verify the label mapping

The notebook assumes binary labels. We inspect the actual values rather than blindly assuming that `1` means depression.


In [ ]:
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    labels = sorted(df[LABEL_COLUMN].unique().tolist())
    print(f"{name} labels: {labels}")

all_labels = sorted(set(train_df[LABEL_COLUMN]) | set(val_df[LABEL_COLUMN]) | set(test_df[LABEL_COLUMN]))

if len(all_labels) != 2:
    raise ValueError(f"Expected exactly 2 labels for binary classification, found: {all_labels}")

LABEL0, LABEL1 = all_labels
print("Binary label mapping in this dataset:", LABEL0, "and", LABEL1)


Train labels: [0, 1]
Validation labels: [0, 1]
Test labels: [0, 1]
Binary label mapping in this dataset: 0 and 1


## 4. Check class balance

A large dataset can still be problematic if one class dominates. Keep the original validation/test sets separate so that evaluation remains meaningful.


In [ ]:
def show_balance(df, name):
    counts = df[LABEL_COLUMN].value_counts().sort_index()
    percentages = (counts / len(df) * 100).round(2)
    result = pd.DataFrame({"count": counts, "percentage": percentages})
    print(f"\n{name}")
    display(result)

show_balance(train_df, "Train")
show_balance(val_df, "Validation")
show_balance(test_df, "Test")



Train


,count,percentage
label,,
0,1389130,80.52
1,336108,19.48



Validation


,count,percentage
label,,
0,297671,80.52
1,72023,19.48



Test


,count,percentage
label,,
0,297672,80.52
1,72023,19.48


## 5. Create a stratified, memory-safe baseline subset

This is the main fix for the previous crash. The old approach could transform the entire multi-million-row training set into TF-IDF. Here we sample at most 200,000 rows while preserving the class distribution.


In [ ]:
baseline_size = min(BASELINE_TRAIN_SIZE, len(train_df))

if baseline_size < len(train_df):
    baseline_df, _ = train_test_split(
        train_df,
        train_size=baseline_size,
        stratify=train_df[LABEL_COLUMN],
        random_state=SEED
    )
else:
    baseline_df = train_df.copy()

baseline_df = baseline_df.reset_index(drop=True)

X_train_baseline = baseline_df[TEXT_COLUMN].fillna("")
y_train_baseline = baseline_df[LABEL_COLUMN]

X_val = val_df[TEXT_COLUMN].fillna("")
y_val = val_df[LABEL_COLUMN]

X_test = test_df[TEXT_COLUMN].fillna("")
y_test = test_df[LABEL_COLUMN]

print("Baseline training rows:", len(baseline_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))


Baseline training rows: 200000
Validation rows: 369694
Test rows: 369695


# Part A — Classical Baseline: TF-IDF + LinearSVC

We use LinearSVC instead of converting the sparse TF-IDF matrix to a dense NumPy array. This is substantially more memory-efficient.


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_tfidf = vectorizer.fit_transform(X_train_baseline)
print("TF-IDF train shape:", X_train_tfidf.shape)

X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF validation shape:", X_val_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


TF-IDF train shape: (200000, 20000)
TF-IDF validation shape: (369694, 20000)
TF-IDF test shape: (369695, 20000)


In [ ]:
svm = LinearSVC(
    class_weight="balanced",
    max_iter=3000,
    random_state=SEED
)

svm.fit(X_train_tfidf, y_train_baseline)
print("LinearSVC training completed.")


LinearSVC training completed.


In [ ]:
def evaluate_binary_model(name, y_true, y_pred, decision_scores=None):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=LABEL1, zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n===== {name} =====")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if decision_scores is not None:
        try:
            auc = roc_auc_score(y_true, decision_scores)
            print(f"ROC-AUC  : {auc:.4f}")
        except Exception as e:
            print("ROC-AUC unavailable:", e)

    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred,
        labels=[LABEL0, LABEL1],
        target_names=[f"Class {LABEL0}", f"Class {LABEL1}"],
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[LABEL0, LABEL1]))

    return {
        "model": name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

svm_val_pred = svm.predict(X_val_tfidf)
svm_val_scores = svm.decision_function(X_val_tfidf)

svm_val_results = evaluate_binary_model(
    "TF-IDF + LinearSVC (Validation)",
    y_val,
    svm_val_pred,
    svm_val_scores
)



===== TF-IDF + LinearSVC (Validation) =====
Accuracy : 0.9333
Precision: 0.7898
Recall   : 0.8962
F1-score : 0.8397
ROC-AUC  : 0.9746

Classification report:
              precision    recall  f1-score   support

     Class 0       0.97      0.94      0.96    297671
     Class 1       0.79      0.90      0.84     72023

    accuracy                           0.93    369694
   macro avg       0.88      0.92      0.90    369694
weighted avg       0.94      0.93      0.93    369694

Confusion matrix:
[[280496  17175]
 [  7477  64546]]


In [ ]:
svm_test_pred = svm.predict(X_test_tfidf)
svm_test_scores = svm.decision_function(X_test_tfidf)

svm_test_results = evaluate_binary_model(
    "TF-IDF + LinearSVC (Test)",
    y_test,
    svm_test_pred,
    svm_test_scores
)



===== TF-IDF + LinearSVC (Test) =====
Accuracy : 0.9345
Precision: 0.7932
Recall   : 0.8976
F1-score : 0.8422
ROC-AUC  : 0.9754

Classification report:
              precision    recall  f1-score   support

     Class 0       0.97      0.94      0.96    297672
     Class 1       0.79      0.90      0.84     72023

    accuracy                           0.93    369695
   macro avg       0.88      0.92      0.90    369695
weighted avg       0.94      0.93      0.94    369695

Confusion matrix:
[[280816  16856]
 [  7373  64650]]


In [ ]:
# # Free large sparse matrices before Transformer training
# del X_train_tfidf, X_val_tfidf, X_test_tfidf
# del vectorizer
# gc.collect()

# print("TF-IDF matrices released from memory.")


# Free large sparse matrices before Transformer training
# (keep the vectorizer itself — small object, needed later for fair comparison)
del X_train_tfidf, X_val_tfidf, X_test_tfidf
gc.collect()

print("Large TF-IDF matrices released. Vectorizer kept for later fair comparison.")


Large TF-IDF matrices released. Vectorizer kept for later fair comparison.


# Part B — Transformer Fine-Tuning

The Transformer is trained separately from the classical baseline. We use a manageable stratified subset to avoid unnecessary RAM/GPU pressure.

If your GPU is small, reduce `TRANSFORMER_TRAIN_SIZE` to 50,000 and/or `MAX_LENGTH` to 128.


In [ ]:
# Optional package imports
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [ ]:
def stratified_subset(df, n):
    n = min(n, len(df))
    if n == len(df):
        return df.copy().reset_index(drop=True)

    subset, _ = train_test_split(
        df,
        train_size=n,
        stratify=df[LABEL_COLUMN],
        random_state=SEED
    )
    return subset.reset_index(drop=True)

transformer_train_df = stratified_subset(train_df, TRANSFORMER_TRAIN_SIZE)
transformer_val_df = stratified_subset(val_df, TRANSFORMER_VAL_SIZE)
transformer_test_df = stratified_subset(test_df, TRANSFORMER_TEST_SIZE)

print(transformer_train_df.shape)
print(transformer_val_df.shape)
print(transformer_test_df.shape)


(100000, 2)
(20000, 2)
(20000, 2)


## 6. Convert labels to 0/1

The Hugging Face classifier expects class IDs from `0` to `num_labels - 1`. This cell creates an explicit mapping from the dataset's original binary labels.


In [ ]:
label2id = {LABEL0: 0, LABEL1: 1}
id2label = {0: str(LABEL0), 1: str(LABEL1)}

for df in [transformer_train_df, transformer_val_df, transformer_test_df]:
    df["labels"] = df[LABEL_COLUMN].map(label2id).astype(int)

print("label2id:", label2id)
print("id2label:", id2label)


label2id: {0: 0, 1: 1}
id2label: {0: '0', 1: '1'}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = Dataset.from_pandas(
    transformer_train_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    transformer_val_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_ds = Dataset.from_pandas(
    transformer_test_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=[TEXT_COLUMN])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=[TEXT_COLUMN])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=[TEXT_COLUMN])

print(train_ds)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100000
})


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: str(LABEL0), 1: str(LABEL1)},
    label2id={str(LABEL0): 0, str(LABEL1): 1}
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(model.config.id2label)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: '0', 1: '1'}


## 7. Transformer metrics

For depression-risk classification, F1 and recall should be reported alongside accuracy. Accuracy alone can hide poor performance on the minority class.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    try:
        probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        auc = roc_auc_score(labels, probabilities)
    except Exception:
        auc = 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc
    }


In [ ]:
# Compute class weights (inverse frequency — same idea as sklearn's "balanced")
from torch.nn import CrossEntropyLoss

class_counts = transformer_train_df["labels"].value_counts().sort_index()
total = class_counts.sum()
class_weights = torch.tensor(
    [total / (2 * class_counts[0]), total / (2 * class_counts[1])],
    dtype=torch.float
).to(DEVICE)

print("Class weights [class0, class1]:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

Class weights [class0, class1]: tensor([0.6210, 2.5665])


In [ ]:
output_dir = "./depression_transformer"

training_args = TrainingArguments(
    output_dir=output_dir,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    # Safer than batch size 16 for smaller GPUs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    # Effective batch size becomes 16
    gradient_accumulation_steps=2,

    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",

    seed=SEED
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,   # <-- renamed from `tokenizer`
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created.")


Trainer created.


In [ ]:
# Clear unused objects before GPU training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
print("Transformer training completed.")


Epoch,Training Loss,Validation Loss


In [ ]:
val_metrics = trainer.evaluate(eval_dataset=val_ds)
print("Validation metrics:")
print(val_metrics)


In [ ]:
test_output = trainer.predict(test_ds)

test_logits = test_output.predictions
test_labels = test_output.label_ids
test_pred = np.argmax(test_logits, axis=-1)

test_probabilities = torch.softmax(
    torch.tensor(test_logits), dim=-1
).numpy()[:, 1]

transformer_test_results = evaluate_binary_model(
    "Transformer (Test)",
    test_labels,
    test_pred,
    test_probabilities
)


In [ ]:
# Fair comparison: re-evaluate SVM on the SAME rows the transformer was tested on
X_test_matched = vectorizer.transform(transformer_test_df[TEXT_COLUMN].fillna(""))
y_test_matched = transformer_test_df[LABEL_COLUMN]

svm_matched_pred = svm.predict(X_test_matched)
svm_matched_scores = svm.decision_function(X_test_matched)

svm_matched_results = evaluate_binary_model(
    "TF-IDF + LinearSVC (Matched Test Subset)",
    y_test_matched,
    svm_matched_pred,
    svm_matched_scores
)

# Part C — Compare the Models


In [ ]:
# comparison = pd.DataFrame([
#     svm_test_results,
#     transformer_test_results
# ])

# comparison

comparison = pd.DataFrame([
    svm_matched_results,       # <- use this instead of svm_test_results
    transformer_test_results
])
comparison


## Interpretation

Use this table in the report to compare the classical and Transformer approaches.

A Transformer should not be claimed to be "better" simply because it is more advanced. The conclusion should be based on the measured test-set metrics, especially F1, recall and ROC-AUC, while also considering computational cost and explainability.


# Part D — SHAP Explainability

SHAP is applied to selected examples rather than the entire test set. Explaining thousands of Transformer predictions can be extremely expensive.

For the Transformer, a text classification prediction function is created and SHAP's text masker is used to estimate the contribution of tokens.


In [ ]:
# Install if necessary:
# !pip -q install shap

import shap
import matplotlib.pyplot as plt

print("SHAP version:", shap.__version__)


In [ ]:
# Put the model in evaluation mode
model.eval()

def predict_proba_text(texts):
    if isinstance(texts, str):
        texts = [texts]

    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probabilities = torch.softmax(outputs.logits, dim=-1)

    return probabilities.detach().cpu().numpy()

print(predict_proba_text(["I feel hopeless and have lost interest in everything."]))


In [ ]:
# Select a small number of test examples for explanation
N_SHAP = 5

sample_n = min(N_SHAP, len(transformer_test_df))

shap_sample = transformer_test_df.sample(
    n=sample_n,
    random_state=SEED
)

sample_texts = shap_sample[TEXT_COLUMN].tolist()

for i, text in enumerate(sample_texts, start=1):
    print(f"Example {i}:")
    print(text[:500])
    print("-" * 80)


In [ ]:
# Text masker and SHAP explainer
masker = shap.maskers.Text(tokenizer)

explainer = shap.Explainer(
    predict_proba_text,
    masker
)

shap_values = explainer(sample_texts)

print("SHAP explanation generated.")


In [ ]:
# # Explain the positive class (class ID 1)
# # Depending on the installed SHAP version, this indexing may be represented
# # as output dimension 1.

# try:
#     shap.plots.text(shap_values[:, :, 1])
# except Exception as e:
#     print("SHAP text plot indexing issue:", e)
#     print("Trying the first output dimension...")
#     shap.plots.text(shap_values[:, :, 0])



print("SHAP version:", shap.__version__)
print("shap_values.output_names:", getattr(shap_values, "output_names", "N/A"))

# Explicitly find the index for class 1 (depression) instead of guessing
if hasattr(shap_values, "output_names") and shap_values.output_names is not None:
    class1_idx = list(shap_values.output_names).index(str(LABEL1)) if str(LABEL1) in shap_values.output_names else 1
else:
    class1_idx = 1  # fallback: predict_proba_text returns [P(class0), P(class1)], so index 1 = class 1

print(f"Using output index {class1_idx} for class '{LABEL1}' (depression)")
shap.plots.text(shap_values[:, :, class1_idx])

## Reading SHAP explanations

For the selected class, positive token contributions indicate text that pushes the model toward that class, while negative contributions push it away.

These explanations are model explanations, not evidence that individual words clinically indicate depression.


# Part E — Optional: Inspect Important TF-IDF/SVM Features

This is a second, simpler form of explainability for the classical baseline. It is useful in the report because it lets you compare global SVM feature weights with local Transformer SHAP explanations.


In [ ]:
# Re-create a small TF-IDF model only if you want feature inspection.
# This avoids keeping the earlier large sparse matrices in memory.

feature_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    sublinear_tf=True,
    dtype=np.float32
)

X_feature_train = feature_vectorizer.fit_transform(
    X_train_baseline
)

feature_svm = LinearSVC(
    class_weight="balanced",
    max_iter=3000,
    random_state=SEED
)

feature_svm.fit(X_feature_train, y_train_baseline)

feature_names = np.array(feature_vectorizer.get_feature_names_out())
weights = feature_svm.coef_[0]

top_positive = np.argsort(weights)[-20:][::-1]
top_negative = np.argsort(weights)[:20]

print("Top features pushing toward class 1:")
for idx in top_positive:
    print(feature_names[idx], round(weights[idx], 4))

print("\nTop features pushing toward class 0:")
for idx in top_negative:
    print(feature_names[idx], round(weights[idx], 4))

del X_feature_train
del feature_vectorizer
del feature_svm
gc.collect()


# Part F — Save the Transformer

Saving the model and tokenizer allows later inference without retraining.


In [ ]:
SAVE_DIR = "./final_depression_transformer"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)


# Part G — Simple Inference Function


In [ ]:
def predict_depression(text):
    model.eval()

    # Apply the SAME preprocessing used during training
    cleaned_text = clean_text(text)

    encoded = tokenizer(
        cleaned_text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    predicted_id = int(np.argmax(probabilities))

    return {
        "predicted_label_id": predicted_id,
        "predicted_label": id2label[predicted_id],
        "probability_class_0": float(probabilities[0]),
        "probability_class_1": float(probabilities[1])
    }

# Final Research Notes

### Recommended experimental statement

The classical model serves as a computationally efficient baseline, while the fine-tuned Transformer is evaluated as the main contextual language model. SHAP provides local explanations for selected predictions.

### Important limitations

- Reddit/social-media labels may not represent clinical diagnoses.
- Dataset construction and label quality can introduce bias.
- Posts can contain sarcasm, slang, quotations, or ambiguous language.
- User-level leakage must be avoided if multiple posts from the same user exist.
- Explainability methods show model behavior; they do not establish clinical causation.
- Do not deploy the model as a medical diagnosis system.

### Main memory fixes implemented

1. Only required CSV columns are loaded.
2. Missing/empty text is removed.
3. TF-IDF is trained on a stratified subset.
4. TF-IDF matrices remain sparse.
5. Large matrices are explicitly deleted after the baseline.
6. Transformer training uses a manageable subset.
7. Transformer batch size starts at 8 instead of 16.
8. Gradient accumulation provides a larger effective batch size.
9. SHAP explanations are generated only for a small sample.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/depression_detector_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to Drive.")

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = "/content/drive/MyDrive/depression_detector_model"  # your saved path

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
model.eval()

MAX_LENGTH = 256
id2label = model.config.id2label

def predict_ui(text):
    if not text or not text.strip():
        return "Please enter some text.", {}

    encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)[0].numpy()

    label = id2label[int(probs.argmax())]
    label_display = "⚠️ Depression-like" if label == "1" else "✅ Normal"

    confidence_dict = {
        "Normal": float(probs[0]),
        "Depression-like": float(probs[1])
    }

    return label_display, confidence_dict

In [ ]:
demo = gr.Interface(
    fn=predict_ui,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type or paste a social media post here...",
        label="Input Text"
    ),
    outputs=[
        gr.Label(label="Prediction"),
        gr.Label(label="Confidence Scores", num_top_classes=2)
    ],
    title="Explainable Depression Detection — Research Demo",
    description=(
        "⚠️ This is a research prototype for academic purposes only. "
        "It classifies text based on patterns learned from Reddit posts and is "
        "**not a diagnostic tool**. It should not be used to self-diagnose or "
        "diagnose others. If you or someone you know is struggling, please reach "
        "out to a mental health professional or a helpline."
    ),
    examples=[
        ["I feel hopeless and empty inside."],
        ["Just got back from an amazing hike with friends, feeling great!"],
        ["Nothing feels worth doing anymore, I just want to stay in bed all day."]
    ]
)

demo.launch(share=True)

In [ ]:
import shap

masker = shap.maskers.Text(tokenizer)

def predict_proba_for_shap(texts):
    cleaned = [clean_text(t) for t in texts]              # <- FIX: preprocess first
    encoded = tokenizer(cleaned, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        return torch.softmax(model(**encoded).logits, dim=-1).numpy()

explainer_ui = shap.Explainer(predict_proba_for_shap, masker)

def predict_with_explanation(text):
    cleaned_text = clean_text(text)                        # <- FIX: preprocess before prediction too

    encoded = tokenizer(cleaned_text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)[0].numpy()

    label = id2label[int(probs.argmax())]
    label_display = "⚠️ Depression-like" if label == "1" else "✅ Normal"
    confidence_dict = {"Normal": float(probs[0]), "Depression-like": float(probs[1])}

    shap_values = explainer_ui([cleaned_text])              # <- FIX: explain the cleaned text, not raw
    html = shap.plots.text(shap_values[:, :, 1], display=False)

    return label_display, confidence_dict, html

demo_with_shap = gr.Interface(
    fn=predict_with_explanation,
    inputs=gr.Textbox(lines=5, placeholder="Type text here...", label="Input Text"),
    outputs=[
        gr.Label(label="Prediction"),
        gr.Label(label="Confidence Scores"),
        gr.HTML(label="Word-level Explanation (SHAP)")
    ],
    title="Explainable Depression Detection — with SHAP Explanation",
    description="Shows which words influenced the prediction, alongside the result.",
)

demo_with_shap.launch(share=True)

Explainable Model


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, shap

SAVE_DIR = "/content/drive/MyDrive/depression_detector_model"  # your saved path

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
model.eval()

MAX_LENGTH = 256

In [ ]:
def predict_proba_text(texts):
    cleaned = [clean_text(t) for t in texts]
    encoded = tokenizer(cleaned, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        return torch.softmax(model(**encoded).logits, dim=-1).numpy()
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_proba_text, masker)

def predict_and_explain(text):
    # 1. Prediction
    probs = predict_proba_text([text])[0]
    label = "Depression-like" if probs[1] > 0.5 else "Normal"
    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence -> Normal: {probs[0]:.2%} | Depression-like: {probs[1]:.2%}")
    print("-" * 70)

    # 2. Explanation (word-level, why the model predicted this)
    shap_values = explainer([text])
    shap.plots.text(shap_values[:, :, 1])   # 1 = depression class index

In [ ]:
import joblib, json, os

ARTIFACT_DIR = "/content/drive/MyDrive/depression_detector_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the transformer model + tokenizer (if not already saved)
trainer.save_model(f"{ARTIFACT_DIR}/model")
tokenizer.save_pretrained(f"{ARTIFACT_DIR}/model")

# Save the TF-IDF vectorizer + SVM (needed for word-frequency dashboard plot)
joblib.dump(vectorizer, f"{ARTIFACT_DIR}/vectorizer.pkl")
joblib.dump(feature_svm, f"{ARTIFACT_DIR}/feature_svm.pkl")

# Save evaluation metrics as plain JSON
with open(f"{ARTIFACT_DIR}/metrics.json", "w") as f:
    json.dump({
        "svm_matched_results": {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                                  for k, v in svm_matched_results.items()},
        "transformer_test_results": {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                                       for k, v in transformer_test_results.items()},
        "confusion_matrix": [[int(x) for x in row] for row in
                              [[15622, 482], [421, 3475]]]  # replace with your actual test_output confusion matrix if different
    }, f, indent=2)

# Save a small sample of the cleaned dataset (for class distribution / text length plots)
# Full 2.47M rows isn't needed — a representative sample is enough and much lighter to reload
df_sample = df[['text_clean', 'label']].sample(n=min(50000, len(df)), random_state=SEED)
df_sample.to_csv(f"{ARTIFACT_DIR}/df_sample.csv", index=False)

print("All artifacts saved to:", ARTIFACT_DIR)
print(os.listdir(ARTIFACT_DIR))

In [ ]:
# ============================================================
# Explainable Depression Detection — Full Gradio App
# ============================================================

!pip -q install gradio shap

import gradio as gr
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib, json, re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ARTIFACT_DIR = "/content/drive/MyDrive/depression_detector_artifacts"
MAX_LENGTH = 256

# ---------- Load model + tokenizer ----------
tokenizer = AutoTokenizer.from_pretrained(f"{ARTIFACT_DIR}/model")
model = AutoModelForSequenceClassification.from_pretrained(f"{ARTIFACT_DIR}/model")
model.eval()
id2label = model.config.id2label

# ---------- Load SVM artifacts (for word frequency plot) ----------
vectorizer = joblib.load(f"{ARTIFACT_DIR}/vectorizer.pkl")
feature_svm = joblib.load(f"{ARTIFACT_DIR}/feature_svm.pkl")

# ---------- Load metrics ----------
with open(f"{ARTIFACT_DIR}/metrics.json") as f:
    saved_metrics = json.load(f)
svm_matched_results = saved_metrics["svm_matched_results"]
transformer_test_results = saved_metrics["transformer_test_results"]
cm = np.array(saved_metrics["confusion_matrix"])

# ---------- Load dataset sample ----------
df = pd.read_csv(f"{ARTIFACT_DIR}/df_sample.csv")

# ---------- Preprocessing (MUST match training exactly) ----------
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'[*_~`>#]', ' ', text)
    text = re.sub(r'&amp;|&gt;|&lt;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ---------- SHAP explainer ----------
masker = shap.maskers.Text(tokenizer)

def predict_proba_text(texts):
    cleaned = [clean_text(t) for t in texts]
    encoded = tokenizer(cleaned, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        return torch.softmax(model(**encoded).logits, dim=-1).numpy()

explainer_ui = shap.Explainer(predict_proba_text, masker)

# ---------- TAB 1: Predict + Explain ----------
def predict_and_explain_ui(text):
    if not text or not text.strip():
        return "Please enter some text.", {}, "<p>No explanation available.</p>"

    cleaned_text = clean_text(text)
    encoded = tokenizer(cleaned_text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)[0].numpy()

    label = id2label[int(probs.argmax())]
    label_display = "⚠️ Depression-like" if str(label) == "1" else "✅ Normal"
    confidence_dict = {"Normal": float(probs[0]), "Depression-like": float(probs[1])}

    shap_values = explainer_ui([cleaned_text])
    html = shap.plots.text(shap_values[:, :, 1], display=False)

    return label_display, confidence_dict, html

# ---------- TAB 2: Dashboard plots ----------
def class_distribution_plot():
    counts = df['label'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.pie(counts, labels=['Normal (0)', 'Depression (1)'], autopct='%1.1f%%',
           colors=['#4C72B0', '#DD8452'], startangle=90)
    ax.set_title("Dataset Class Distribution")
    plt.tight_layout()
    return fig

def word_frequency_plot(top_n=15):
    coefs = feature_svm.coef_[0]
    feature_names = np.array(vectorizer.get_feature_names_out())
    top_pos_idx = np.argsort(coefs)[-top_n:][::-1]
    top_neg_idx = np.argsort(coefs)[:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].barh(feature_names[top_pos_idx][::-1], coefs[top_pos_idx][::-1], color='#DD8452')
    axes[0].set_title("Top Words → Depression Class")
    axes[0].set_xlabel("SVM Weight")
    axes[1].barh(feature_names[top_neg_idx][::-1], coefs[top_neg_idx][::-1], color='#4C72B0')
    axes[1].set_title("Top Words → Normal Class")
    axes[1].set_xlabel("SVM Weight")
    plt.tight_layout()
    return fig

def model_comparison_plot():
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    svm_vals = [svm_matched_results[m] for m in metrics]
    tr_vals = [transformer_test_results[m] for m in metrics]
    x = np.arange(len(metrics))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width/2, svm_vals, width, label='TF-IDF + SVM', color='#4C72B0')
    ax.bar(x + width/2, tr_vals, width, label='DistilBERT', color='#DD8452')
    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1)
    ax.set_title("Baseline vs Transformer — Matched Test Set")
    ax.legend()
    plt.tight_layout()
    return fig

def confusion_matrix_plot():
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred: Normal", "Pred: Depression"])
    ax.set_yticklabels(["True: Normal", "True: Depression"])
    ax.set_title("Transformer — Confusion Matrix")
    plt.tight_layout()
    return fig

def text_length_distribution_plot():
    df['text_length'] = df['text_clean'].astype(str).str.len()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df[df['label']==0]['text_length'], bins=50, alpha=0.6, label='Normal', color='#4C72B0', range=(0,2000))
    ax.hist(df[df['label']==1]['text_length'], bins=50, alpha=0.6, label='Depression', color='#DD8452', range=(0,2000))
    ax.set_xlabel("Post Length (characters)")
    ax.set_ylabel("Count")
    ax.set_title("Text Length Distribution by Class")
    ax.legend()
    plt.tight_layout()
    return fig

def build_dashboard_text():
    metrics_md = f"""
### Model Performance (Test Set, N=20,000, matched)

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| TF-IDF + LinearSVC (baseline) | {svm_matched_results['accuracy']:.4f} | {svm_matched_results['precision']:.3f} | {svm_matched_results['recall']:.3f} | {svm_matched_results['f1']:.3f} | {svm_matched_results.get('roc_auc', 0):.3f} |
| **DistilBERT (fine-tuned)** | **{transformer_test_results['accuracy']:.4f}** | **{transformer_test_results['precision']:.3f}** | **{transformer_test_results['recall']:.3f}** | **{transformer_test_results['f1']:.3f}** | **{transformer_test_results.get('roc_auc', 0):.3f}** |
"""
    limitations_md = """
### Known Limitation: Lexical Shortcut on "interest"

SHAP-based error analysis revealed the model under-detects anhedonia symptoms phrased as **"lost interest"**, even though it reliably catches other high-intensity depressive language.

| Test sentence | Confidence (Depression-like) |
|---|---|
| "I lost interest in everything." | 0.55% |
| "I have a lot of interest in everything." | 0.11% |
| "I lost my job and my friends." | 71.34% |
| "I feel hopeless and empty inside." | 99.43% |

**Root cause:** the token *"interest"* carries a strong standalone negative-attribution bias, largely independent of context — a lexical shortcut rather than true compositional understanding.
"""
    return metrics_md, limitations_md

# ---------- Build the app ----------
with gr.Blocks(title="Explainable Depression Detection") as demo:
    gr.Markdown("# Explainable Depression Detection — Research Demo")
    gr.Markdown(
        "⚠️ **Disclaimer:** This is a research prototype for academic purposes only. "
        "It is **not a diagnostic tool** and should not be used to self-diagnose or "
        "diagnose others. If you or someone you know is struggling, please reach out "
        "to a mental health professional or a helpline."
    )

    with gr.Tabs():
        with gr.Tab("🔍 Predict"):
            gr.Markdown("Enter a social media–style post to classify and explain.")
            text_input = gr.Textbox(lines=5, placeholder="Type or paste text here...", label="Input Text")
            predict_btn = gr.Button("Analyze", variant="primary")
            pred_label = gr.Label(label="Prediction")
            pred_confidence = gr.Label(label="Confidence Scores")
            pred_explanation = gr.HTML(label="Word-level Explanation (SHAP)")

            predict_btn.click(
                fn=predict_and_explain_ui,
                inputs=text_input,
                outputs=[pred_label, pred_confidence, pred_explanation]
            )

            gr.Examples(
                examples=[
                    ["I feel hopeless and empty inside."],
                    ["Just got back from an amazing hike with friends, feeling great!"],
                    ["I've been feeling really low lately and lost interest in everything."]
                ],
                inputs=text_input
            )

        with gr.Tab("📊 Analysis Dashboard"):
            gr.Markdown("Model evaluation results and dataset analysis.")
            metrics_text, limitations_text = build_dashboard_text()
            gr.Markdown(metrics_text)

            with gr.Row():
                gr.Plot(value=model_comparison_plot())
                gr.Plot(value=confusion_matrix_plot())

            with gr.Row():
                gr.Plot(value=class_distribution_plot())
                gr.Plot(value=text_length_distribution_plot())

            gr.Markdown("### Word Frequency Analysis (Global Feature Importance)")
            gr.Plot(value=word_frequency_plot())

            gr.Markdown(limitations_text)

demo.launch(share=True)